# 03 — Equity underlying

`tq.Equity`: one view on the pricing data of an underlying (spot, repo, dividends, vol, discount curve), forward and its sensitivities. Q/A: `docs/chapters/equity.tex` (Forwards and dividends).

In [ ]:
import os, sys
from datetime import timedelta
from pathlib import Path
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

REPO = Path.cwd().resolve().parents[1]

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorquant as tq


## Market and underlying

In [ ]:
mkt = tq.MarketEnvironment.from_folder(REPO / "marketdata")   # latest snapshot
EVALUATION_DATE = tq.Settings.evaluation_date = mkt.evaluation_date

eq = mkt.get_equity("TICKER1")
expiry = EVALUATION_DATE + timedelta(days=365)
eq

In [ ]:
eq.summary(expiry)

A view, not a copy: same `tf.Variable`s as the market.

In [ ]:
print(eq.spot is mkt.get_eq_spot("TICKER1"), eq.vol_surface is mkt.get_eq_vol_surface("TICKER1"))

## Forward term structure
Discrete dividends: $F(T) = (S_0 - PV_{div}(T))\,e^{-q_{repo}T}/P(0,T)$ — the forward drops at each ex-date.

In [ ]:
days = np.arange(1, 2 * 365 + 1, 3)
dates = [EVALUATION_DATE + timedelta(days=int(d)) for d in days]
T = days / 365
fwd = [float(eq.forward(d)) for d in dates]
no_div = [float(eq.spot) * np.exp(-float(eq.repo) * t) / float(eq.discount_curve.discount(d)) for t, d in zip(T, dates)]

plt.figure(figsize=(7, 4))
plt.plot(T, no_div, "--", label="no dividends")
plt.plot(T, fwd, label="with discrete dividends")
for ex in eq.dividends.ex_dates:
    plt.axvline((ex - EVALUATION_DATE).days / 365, color="grey", lw=0.8, ls=":")
plt.xlabel("T (years)"); plt.ylabel("forward"); plt.title("TICKER1 forward"); plt.grid(alpha=0.3); plt.legend()

## Implied dividend yield
$q_{eff}(T) = -\ln(1 - PV_{div}(T)/S_0)/T$: the continuous yield giving the same forward. It is not flat: steps at ex-dates, then decays.

In [ ]:
q_eff = [float(eq.implied_div_yield(d)) * 100 for d in dates]
plt.figure(figsize=(7, 4))
plt.plot(T, q_eff)
plt.xlabel("T (years)"); plt.ylabel("q_eff (%)"); plt.title("Implied dividend yield"); plt.grid(alpha=0.3)

## Sensitivities of the forward (AAD)
One tape, gradients on the underlying's risk factors. Check: $\partial F/\partial S = e^{-q_{repo}T}/P(0,T)$ (cash dividends do not move with $S$), $\partial F/\partial q_{repo} = -T\,F$, $\partial F/\partial D_i = -P(0,ex_i)\,e^{-q_{repo}T}/P(0,T)$ for $ex_i \le T$ (0 after).

In [ ]:
rf = eq.risk_factors()
with tf.GradientTape() as tape:
    F = eq.forward(expiry)
grads = tape.gradient(F, rf)

t = eq.year_fraction(expiry)
P = float(eq.discount_curve.discount(expiry))
print(f"dF/dS    = {float(grads['EQ:EUR:TICKER1:SPOT'][0]):.6f}   closed form {np.exp(-float(eq.repo) * t) / P:.6f}")
print(f"dF/drepo = {float(grads['EQ:EUR:TICKER1:REPO'][0]):.6f}   closed form {-t * float(F):.6f}")
curve_key = eq.discount_curve.name
for pillar, g in zip(eq.discount_curve.pillars, grads[curve_key]):
    print(f"dF/dr({pillar:>5.2f}Y) = {0.0 if g is None else float(g):10.4f}")

dF_dD = grads["EQ:EUR:TICKER1:DIV"][0].numpy()
for ex, amount, g in zip(eq.dividends.ex_dates, eq.dividends.amounts, dF_dD):
    closed = -float(eq.discount_curve.discount(ex)) * np.exp(-float(eq.repo) * t) / P if ex <= expiry else 0.0
    print(f"dF/dD({ex}, D={amount:7.4f}) = {g:9.6f}   closed form {closed:9.6f}")

Bumping the market variable moves the view: $\Delta F \approx \partial F/\partial S \cdot \Delta S$.

In [ ]:
spot = mkt.get_eq_spot("TICKER1")
s0 = float(spot)
spot.assign(s0 + 1.0)
print(f"F(S0+1) - F(S0) = {float(eq.forward(expiry)) - float(F):.6f}")
spot.assign(s0)

## Exercises
1. Compare `eq.forward(expiry)` with the forward of `BlackScholesPricer` (`option.forward`) for both dividend models: why do they coincide?
2. Which pillar sensitivities are non-zero, and why are only those?
3. Build a market without `EQ:EUR:TICKER1:DIV`: which dividend model does `get_equity` pick, and how does $\partial F/\partial S$ change?